# EXP-011 (red-team): local lanes, FCO/FCG context, antigentic admission — matched statistics

## 1 Requirements and predeclared hypothesis
H1-H6 as written in experiments/CLAUDE_LAB_NOTEBOOK.md R02 before heldout results were read. Alpha 0.05, exact two-sided McNemar, 10,000 stratified paired bootstrap replicates (numpy PCG64 seed 20261102), Holm over the predeclared heldout primary family.

## 2 Input source, license, hashes
evidence/redteam/lanes/*_PUBLIC.jsonl (hashed case keys + booleans; LongevityBench CC BY-NC source and gold stay on magicSTUDIObox), *_META.json run receipts, evidence/freeze/longebench_split_manifest.json (sha 46c48f08...).

## 3 Frozen settings
temperature 0, seed 20261102, num_predict 16, num_ctx 4096; Insilico greedy float32 MPS; model digests in META.

## 4 Deterministic preprocessing
keys validated against the frozen partition; duplicates/split mismatch/oracle fields fail closed.

## 5 Model/harness execution
replayed from receipts produced on magicSTUDIObox by scripts/redteam/redteam_local_lanes.py and redteam_insilico_lane.py; no inference here.

## 6 Analysis
paired tables, McNemar, bootstrap, Holm; selective metrics (coverage, false admission, admitted-injected); token/latency totals.

## 7 Negative controls
forged_verify and edge_removed must be 100% ABSTAIN with zero model calls; loader rejects tampered exports.

## 8 Outputs
exp011_lane_stats.json, exp011_attribution_rows.json.

## 9 FCO/FCG dependency changes
adds EXECUTED edges MODEL_EVAL_OF (frozen cases -> lane results) and HARNESS_ABLATION_OF; no predecessor leaf changes.

## 10 Claim ceiling
Notebook PASS verifies the local replay only. It never upgrades NOT_TESTED, BLOCKED or NEGATIVE states, and it never attributes a causal gain to a hash, leaf or edge: hashes bind bytes, controlled interventions bind effects. n=38/39 per split: power to detect small effects is low; a null is not evidence of equivalence.


In [1]:
import json, hashlib, sys
from pathlib import Path
import os
R = Path.cwd(); sys.path.insert(0, str(R / 'src')); sys.path.insert(0, str(R / 'scripts/redteam')); sys.path.insert(0, str(R / 'scripts'))
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def jsonl(p): return [json.loads(x) for x in Path(p).read_text().splitlines() if x.strip()]
OUT = Path(os.environ.get('REDTEAM_RUN_DIR', str(R / 'evidence/notebook_runs_redteam_v1'))) / 'outputs'; OUT.mkdir(parents=True, exist_ok=True)
def emit(name, obj):
    p = OUT / name; p.write_text(json.dumps(obj, sort_keys=True, indent=2) + '\n'); print('OUTPUT', name, sha(p)); return p
from frontier_forms.redteam_stats import (load_public_lane, arm_flags, compare, holm, selective_metrics, base_metrics, admission_rule, injection_metrics)
split_m = json.loads((R / 'evidence/freeze/longebench_split_manifest.json').read_text())
allowed = {s: {k for t, p in split_m['task_partitions'].items() for k in p[s]} for s in ('discovery', 'heldout')}
strata = {k: t for t, p in split_m['task_partitions'].items() for s in ('discovery', 'heldout') for k in p[s]}
LANES = R / 'evidence/redteam/lanes'
runs = {}
for meta in sorted(LANES.glob('*_META.json')):
    m = json.loads(meta.read_text()); tag = m['tag']
    if tag in ('smoke', 'smoke2') or 'forms12' in meta.name: continue
    pub = meta.with_name(meta.name.replace('_META.json', '_PUBLIC.jsonl'))
    rows = load_public_lane(pub, m['split'], allowed[m['split']])
    assert len(rows) == len(allowed[m['split']]) == m['cases'], (meta.name, len(rows))
    runs[(tag, m['split'])] = {'meta': m, 'rows': rows}
print('RUNS', sorted(runs))
out = {'per_arm': {}, 'comparisons': {}, 'integrity': {}, 'replay': {}}
BASE_ARMS = ['direct', 'fco', 'fcg', 'inject_direct', 'forged_noverify']
ADM_ARMS = ['antigence_verdict', 'antigence_swap', 'antigence_full', 'fcg_antigence_verdict', 'fcg_antigence_swap', 'fcg_antigence_full',
            'inject_antigence_verdict', 'inject_antigence_swap', 'inject_antigence_full']
for (tag, split), r in sorted(runs.items()):
    if r['meta'].get('arms') == 'direct': continue
    rows = r['rows']; d = {}
    for a in BASE_ARMS: d[a] = base_metrics(rows, a)
    for a in ADM_ARMS: d[a] = selective_metrics(rows, a)
    d['injection'] = injection_metrics(rows)
    # gate view under attack: among cases the clean fcg arm got right, how many did each gate ADMIT with the injected letter?
    for a in ('inject_antigence_swap', 'inject_antigence_verdict', 'inject_antigence_full'):
        d[a]['flip_admitted_injected'] = sum(1 for x in rows if x['arms']['fcg'].get('strict_correct') and x['arms'][a].get('admitted_injected'))
    for a in ('forged_verify', 'edge_removed'):
        d[a] = {'abstain': sum(1 for x in rows if x['arms'][a]['decision'] == 'ABSTAIN'), 'model_calls': sum(1 for x in rows if x['arms'][a]['model_called']), 'n': len(rows)}
        assert d[a]['abstain'] == len(rows) and d[a]['model_calls'] == 0, ('INTEGRITY_GATE_LEAK', tag, split, a)
    out['per_arm'][f'{tag}:{split}'] = d
# within-lane harness comparisons (same weights): fco vs direct, fcg vs direct; across-lane model comparisons: Lx direct vs L0 direct
for split in ('discovery', 'heldout'):
    fam = {}
    l0 = runs.get(('L0', split))
    for (tag, sp), r in sorted(runs.items()):
        if sp != split or r['meta'].get('arms') == 'direct': continue
        base = arm_flags(r['rows'], 'direct')
        for arm in ('fco', 'fcg'):
            fam[f'{tag}:{arm}_vs_direct'] = compare(base, arm_flags(r['rows'], arm), strata)
        if l0 is not None and tag != 'L0':
            fam[f'{tag}:direct_vs_L0:direct'] = compare(arm_flags(l0['rows'], 'direct'), base, strata)
    h = holm({k: v['p_mcnemar_exact_two_sided'] for k, v in fam.items()})
    for k in fam:
        fam[k]['holm'] = h[k]
        fam[k]['admission'] = admission_rule(fam[k]['mcnemar_gate'] and h[k]['reject'], fam[k]['bootstrap_gate'],
                                             'NOT_TESTED' if split == 'discovery' else 'SEE_HELDOUT_ROW', 'NOT_TESTED')
    out['comparisons'][split] = fam
# cold restart replay of L0 direct on discovery
if ('L0', 'discovery') in runs and ('L0replay', 'discovery') in runs:
    a = arm_flags(runs[('L0', 'discovery')]['rows'], 'direct'); b = arm_flags(runs[('L0replay', 'discovery')]['rows'], 'direct')
    frozen = {x['case_digest_sha256']: x['baseline_correct'] for x in jsonl(R / 'evidence/stats/discovery_pair_flags.jsonl')}
    out['replay'] = {'warm_vs_cold_agreement': sum(1 for k in a if a[k] == b[k]), 'warm_vs_frozen_agreement': sum(1 for k in a if a[k] == frozen[k]),
                     'cold_vs_frozen_agreement': sum(1 for k in b if b[k] == frozen[k]), 'n': len(a),
                     'frozen_correct': sum(frozen.values()), 'warm_correct': sum(a.values()), 'cold_correct': sum(b.values())}
emit('exp011_lane_stats.json', out)
for split, fam in out['comparisons'].items():
    print('==', split)
    for k, v in fam.items():
        print(f"{k:40s} acc {v['accuracy_baseline']:.3f}->{v['accuracy_challenger']:.3f} d={v['delta_abs']:+.3f} b/c={v['table']['baseline_only']}/{v['table']['challenger_only']} p={v['p_mcnemar_exact_two_sided']:.4f} holm={v['holm']['p_holm']:.4f} CI={v['ci95_delta_stratified_bootstrap']} -> {v['admission']}")
print('REPLAY', out['replay'])

KeyError: 'tag'